# 🧪 Sesión 19 · Sanitización de Entradas, Prevención de XSS e Inyecciones

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué haces con el HTML que llega del usuario?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/comentarios  {"texto":"<script>alert(1)</script>"}
HTTP/1.1 201 Created
{"id":1,"texto":"&lt;script&gt;alert(1)&lt;/script&gt;"}

GET /api/comentarios
HTTP/1.1 200 OK
{"comentarios":[{"id":1,"texto":"&lt;script&gt;alert(1)&lt;/script&gt;"}]}

En el cuerpo nunca aparece <script> sin escapar.
```

_Guardar texto escapado es una decisión; escapar al mostrar es otra. En esta sesión hacemos ambas para que el resultado sea seguro._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Escapar HTML antes de guardar o devolver texto.
- Explicar por qué la validación de entrada no sustituye al escape de salida.
- Probar payloads de XSS y verificar que no se ejecutan.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Un comentario normal se guarda → HTTP 201
- [ ] El script se neutraliza → HTTP 201
- [ ] El listado tampoco devuelve scripts → HTTP 200

## 🧩 El reto

Implementa un endpoint de comentarios que neutralice el HTML peligroso y devuelva el texto seguro.

**Requisitos del contrato**

- POST /api/comentarios recibe un texto y responde 201 con el comentario creado.
- El texto se escapa antes de guardarse o devolverse.
- GET /api/comentarios devuelve los comentarios sin etiquetas ejecutables.
- Un comentario normal no se altera.
- La respuesta nunca contiene la cadena <script>.

> **Pista conceptual:** En Python, `html.escape` convierte `<`, `>` y `&` en entidades. Escapar no es lo mismo que eliminar: el usuario ve el texto, el navegador no lo ejecuta.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion19.md` con el payload de prueba y el resultado seguro.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [2]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [3]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="API de comentarios")

comentarios = []

class Comentario(BaseModel):
    texto: str

@app.post("/api/comentarios", status_code=201)
def crear_comentario(datos: Comentario):
    nuevo = {
        "id": len(comentarios) + 1,
        "texto": datos.texto
    }

    comentarios.append(nuevo)
    return nuevo

@app.get("/api/comentarios")
def listar_comentarios():
    return {"comentarios": comentarios}



Writing app.py


In [5]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import html

app = FastAPI(title="API segura de comentarios")

comentarios = []

class Comentario(BaseModel):
    texto: str

@app.post("/api/comentarios", status_code=201)
def crear_comentario(datos: Comentario):
    texto_seguro = html.escape(datos.texto)

    nuevo = {
        "id": len(comentarios) + 1,
        "texto": texto_seguro
    }

    comentarios.append(nuevo)

    return nuevo


@app.get("/api/comentarios")
def listar_comentarios():
    return {"comentarios": comentarios}


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Utilizaré Python con FastAPI para desarrollar una API REST. Trabajaré en Google Colab y utilizaré las librerías FastAPI, Pydantic y html. |
| **Rol** | La IA deberá actuar como un desarrollador backend especializado en seguridad web, ayudándome a implementar una API que proteja los comentarios contra ataques XSS. |
| **Especificación** | Crear dos rutas: POST /api/ comentarios, para recibir y guardar comentarios con código HTTP 201, y GET /api/ comentarios, para mostrar todos los comentarios con código HTTP 200. Se debe utilizar escape HTML para neutralizar etiquetas peligrosas, sin modificar los comentarios normales. |
| **Output** | Espero recibir un archivo app.py con el código completo, organizado y funcional, que permita crear y consultar comentarios de forma segura. También espero que supere los tres casos de prueba del verificador. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [6]:
from verificar import probar_http

casos = [{'descripcion': 'Un comentario normal se guarda',
  'metodo': 'POST',
  'ruta': '/api/comentarios',
  'json': {'texto': 'Muy bueno'},
  'estado': 201,
  'json_contiene': {'id': 1, 'texto': 'Muy bueno'}},
 {'descripcion': 'El script se neutraliza',
  'metodo': 'POST',
  'ruta': '/api/comentarios',
  'json': {'texto': '<script>alert(1)</script>'},
  'estado': 201,
  'cuerpo_contiene': ['&lt;script&gt;'],
  'no_contiene': ['<script>']},
 {'descripcion': 'El listado tampoco devuelve scripts',
  'metodo': 'GET',
  'ruta': '/api/comentarios',
  'estado': 200,
  'cuerpo_contiene': ['&lt;script&gt;'],
  'no_contiene': ['<script>']}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. Un comentario normal se guarda
✅ 02. El script se neutraliza
✅ 03. El listado tampoco devuelve scripts
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion19.md` a `auditorias/auditoria_sesion19.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue lograr que el texto con etiquetas HTML se guardara de forma segura, porque al principio no sabía cómo evitar que se ejecutaran los scripts. Después comprendí que podía utilizar html.escape() para convertir los caracteres peligrosos en texto seguro.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?

   Decidí utilizar una lista llamada comentarios para almacenar temporalmente los comentarios y asignar automáticamente un ID diferente a cada uno. También utilicé una clase de Pydantic para validar que la información recibida tuviera el campo texto.

   - ¿Qué error detectó la verificación que tú no habías previsto?

   Al principio no había considerado que el listado de comentarios también podía devolver etiquetas HTML peligrosas. La verificación me ayudó a entender que no basta con proteger la ruta POST, sino que también se debe comprobar que la ruta GET devuelva el contenido escapado.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
